In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# =========================
# 1. Imports
# =========================
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import lightgbm as lgb

# =========================q
# 2. Load Data
# =========================
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')

# =========================
# 3. Separate target
# =========================
target = 'Irrigation_Need'
X = train.drop(columns=[target, 'id'])
y = train[target]

test_ids = test['id']
X_test = test.drop(columns=['id'])

# =========================
# 4. Encode target
# =========================
target_le = LabelEncoder()
y = target_le.fit_transform(y)

# =========================
# 5. Encode categorical features
# =========================
cat_cols = X.select_dtypes(include='object').columns

for col in cat_cols:
    le = LabelEncoder()
    full_col = pd.concat([X[col], X_test[col]], axis=0)
    le.fit(full_col)
    
    X[col] = le.transform(X[col])
    X_test[col] = le.transform(X_test[col])

# =========================
# 6. Train-validation split
# =========================
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# =========================
# 7. Model
# =========================
model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=3,
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight={0:1, 1:1, 2:5},  # boost "High"
    random_state=42,
    n_jobs=-1
)

# =========================
# 8. Train
# =========================
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='multi_logloss',
    callbacks=[lgb.early_stopping(50)]
)

# =========================
# 9. Predict
# =========================
preds = model.predict(X_test)

# Convert back to labels
preds = target_le.inverse_transform(preds)

# =========================
# 10. Submission
# =========================
submission = pd.DataFrame({
    'id': test_ids,
    'Irrigation_Need': preds
})

submission.to_csv('submission.csv', index=False)

print("✅ Submission file created!")